# Bước 1
## Cài đặt thư viện

In [ ]:
%%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {install_numpy} \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes \
    git+https://github.com/huggingface/transformers \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels

## Load model gốc

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
model_name = "unsloth/gpt-oss-20b"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

## Add LORA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## Thử nghiệm model trước khi train

In [ ]:
messages = [
    {"role": "system", "content": "Bạn là chuyên viên phân tích thống kê dữ liệu kiểm toán nội bộ nghành thẻ tại Ngân hàng. Hãy tập trung phân tích các con số, tỷ lệ, xu hướng thị trường và báo cáo tài chính.", "thinking": None},
    {"role": "user", "content": "Tại sao cần kiểm tra định kỳ các tài khoản người dùng không hoạt động (Inactive accounts)?"},
    {"role": "assistant", "content": "Để giảm thiểu rủi ro các tài khoản cũ bị lợi dụng để truy cập trái phép vào hệ thống."}
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

## Load data để fine tune model

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("json", data_files="audit-card2024.jsonl", split="train")
dataset


In [ ]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
print(dataset[0]['text'])

## Train model

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

## Thử lại sau khi finetune

In [ ]:
messages = [
    {"role": "system", "content": "Bạn là chuyên viên phân tích thống kê dữ liệu kiểm toán nội bộ nghành thẻ tại Ngân hàng. Hãy tập trung phân tích các con số, tỷ lệ, xu hướng thị trường và báo cáo tài chính.", "thinking": None},
    {"role": "user", "content": "Tại sao cần kiểm tra định kỳ các tài khoản người dùng không hoạt động (Inactive accounts)?"},
    {"role": "assistant", "content": "Để giảm thiểu rủi ro các tài khoản cũ bị lợi dụng để truy cập trái phép vào hệ thống."}
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

In [ ]:
model.save_pretrained("finetuned_model")

In [ ]:
if False:
  from unsloth import FastLanguageModel
  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = "finetuned_model", # YOUR MODEL YOU USED FOR TRAINING
      max_seq_length = 1024,
      dtype = None,
      load_in_4bit = True,
  )

messages = [
    {'role': 'system', 'content': 'Bạn là chuyên viên phân tích thống kê dữ liệu kiểm toán nội bộ nghành thẻ tại Ngân hàng. Hãy tập trung phân tích các con số, tỷ lệ, xu hướng thị trường và báo cáo tài chính.', 'thinking': None},
    {"role": "user", "content": "Khung quản trị CNTT phổ biến nhất hiện nay là gì?"},
    {"role": "assistant", "content": "COBIT (Control Objectives for Information and Related Technology)."}
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, streamer = TextStreamer(tokenizer))

# cách 2: Colab

In [ ]:
%%capture
# 1. CÀI ĐẶT TỐI ƯU
# Sử dụng lệnh cài đặt chuẩn của Unsloth, bao gồm cả hỗ trợ cho Colab và các model mới
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Cài đặt các thư viện phụ trợ cần thiết
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes
import os
os.environ["WANDB_DISABLED"] = "true" # Tắt wandb để tránh lỗi đăng nhập không cần thiết

from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. CẤU HÌNH MODEL
# Model: unsloth/gpt-oss-20b là phiên bản tối ưu hóa cho tốc độ và bộ nhớ
max_seq_length = 2048 # Tăng lên 2048 để xử lý ngữ cảnh dài hơn, nếu Colab báo OOM hãy hạ xuống 1024
dtype = None # None sẽ tự động phát hiện Float16/Bfloat16
load_in_4bit = True # Bắt buộc cho model 20B trên Colab free tier (T4 16GB)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "openai/gpt-oss-20b", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # Nếu model là private (gated), bỏ comment và điền token
)

# 3. CẤU HÌNH PEFT (LORA)
# Tối ưu hóa cho hiệu quả: Tăng r lên 16 hoặc 32 để model học tốt hơn kiến thức chuyên ngành
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, # Thường đặt gấp 2 lần r
    lora_dropout = 0, # 0 là tối ưu nhất cho inference
    bias = "none",    
    use_gradient_checkpointing = "unsloth", # Quan trọng: Giảm bộ nhớ VRAM khi train
    random_state = 3407,
    use_rslora = True, # Rank Stabilized LoRA giúp hội tụ tốt hơn
    loftq_config = None, 
)

# 4. CHUẨN BỊ DỮ LIỆU
# Lưu ý: Bạn cần upload file 'audit-card2024.jsonl' lên Colab trước khi chạy
try:
    dataset = load_dataset("json", data_files="audit-card2024.jsonl", split="train")
except FileNotFoundError:
    print("CẢNH BÁO: Không tìm thấy file 'audit-card2024.jsonl'. Hãy upload file lên Colab!")
    # Tạo dữ liệu giả để demo nếu không có file thật
    dataset = load_dataset("json", data_files={"data": [{"messages": [{"role": "user", "content": "Hello"}]}]}, split="data")

from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)
# print(dataset[0]['text']) # Kiểm tra dữ liệu

# 5. HUẤN LUYỆN (TRAINING)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1, # Model 20B to, batch size 1 là an toàn
        gradient_accumulation_steps = 4, # Tổng hiệu ứng batch size = 1x4 = 4
        warmup_steps = 5,
        max_steps = 60, # Số bước train giới hạn để test nhanh. Tăng lên nếu muốn train hết epoch.
        # num_train_epochs = 1, 
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Bắt đầu huấn luyện...")
trainer_stats = trainer.train()

# 6. INFERENCE TỐC ĐỘ CAO (SAU KHI TRAIN)
# Quan trọng: Chuyển sang chế độ inference để tăng tốc độ generate
model = FastLanguageModel.for_inference(model) 

print("\n--- Kết quả Test 1 (Sau khi train) ---")
messages = [
    {"role": "system", "content": "Bạn là chuyên viên phân tích thống kê dữ liệu kiểm toán nội bộ nghành thẻ tại Ngân hàng. Hãy tập trung phân tích các con số, tỷ lệ, xu hướng thị trường và báo cáo tài chính."},
    {"role": "user", "content": "Tại sao cần kiểm tra định kỳ các tài khoản người dùng không hoạt động (Inactive accounts)?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to(model.device)

# Sử dụng TextStreamer để xem kết quả theo thời gian thực
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

# 7. LƯU MODEL
model.save_pretrained("finetuned_gpt_oss_20b") # Lưu model LoRA
tokenizer.save_pretrained("finetuned_gpt_oss_20b")

# 8. TEST LẠI VỚI DỮ LIỆU MỚI
print("\n--- Kết quả Test 2 (Dữ liệu mới) ---")
messages = [
    {'role': 'system', 'content': 'Bạn là chuyên viên phân tích thống kê dữ liệu kiểm toán nội bộ nghành thẻ tại Ngân hàng.'},
    {"role": "user", "content": "Khung quản trị CNTT phổ biến nhất hiện nay là gì?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to(model.device)

_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

# Fix Export GGUF.ipynb theo 2 cách khác:

## **CÁCH 1: Tối ưu cho Model 20B (openai/gpt-oss-20b) trên T4 (KHO - Khó khăn)**
Đây là cách ép buộc máy chạy. Vì RAM quá ít, chúng ta phải áp dụng chiến thuật "Chia để trị":

**Không dùng cmake:** Tốn quá nhiều RAM. Sử dụng pip để cài bản pre-build (đã biên dịch sẵn).
Huấn luyện (Train): Chạy xong rồi Restart Runtime ngay lập tức để giải phóng RAM.
**Export: **Load lại model chỉ để Export, không làm gì khác.
Lưu ý: Tỷ lệ thành công cách này trên T4 12GB RAM chỉ khoảng 60-70%. Nếu bị lỗi "Out of Memory" lúc export, bạn phải dùng cách "Merge Local" (mình sẽ hướng dẫn ở cuối).

**Code Phần 1: Cài đặt và Huấn luyện**
Chạy đoạn này, đợi train xong, quan trọng là Lưu Adapter.

In [ ]:
%%capture
# 1. CÀI ĐẶT (Chỉ dùng pip, không dùng git clone + cmake)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes

import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. LOAD MODEL (T4 GPU 16GB vừa đủ để chạy 20B ở 4-bit)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "openai/gpt-oss-20b",
    max_seq_length = 1024, # Giảm xuống 1024 để tiết kiệm VRAM
    dtype = None,
    load_in_4bit = True,
)

# 3. CẤU HÌNH LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Bắt buộc để tiết kiệm RAM/VRAM
    random_state = 3407,
    use_rslora = True,
)

# 4. LOAD DỮ LIỆU (Giả sử file đã có)
try:
    dataset = load_dataset("json", data_files="audit-card2024.jsonl", split="train")
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    dataset = dataset.map(formatting_prompts_func, batched = True)
except:
    print("Không tìm thấy data, tạo data giả để chạy demo")
    dataset = load_dataset("json", data_files={"data": [{"messages": [{"role": "user", "content": "Test"}]}]}, split="data")

# 4. LOAD DỮ LIỆU (Giả sử file đã có)
try:
    dataset = load_dataset("json", data_files="audit-card2025.jsonl", split="train")
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    dataset = dataset.map(formatting_prompts_func, batched = True)
except:
    print("Không tìm thấy data, tạo data giả để chạy demo")
    dataset = load_dataset("json", data_files={"data": [{"messages": [{"role": "user", "content": "Test"}]}]}, split="data")


# 5. TRAINING
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1, # Batch size 1
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30, # Train ít bước để test nhanh (Tăng lên nếu cần)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()

# 6. LƯU ADAPTER (QUAN TRỌNG)
# Chúng ta chỉ lưu phần train thêm (Adapter) chứ chưa merge GGUF ngay
model.save_pretrained("gpt20b_lora_adapter") 
tokenizer.save_pretrained("gpt20b_lora_adapter")
print("Đã lưu Adapter thành công! Hãy làm Bước Restart Runtime.")

Sau khi chạy xong đoạn trên:

Vào Menu Runtime -> Restart session (hoặc Restart Runtime).
Làm tiếp Code Phần 2 ở ô mới.

**Code Phần 2:** Restart xong -> Export GGUF
Phải chạy lại cell import sau khi restart.

In [ ]:
%%capture
# Cài lại thư viện cơ bản sau khi restart
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Cài llama-cpp-python (đây là bản nhẹ, không cần cmake)
!pip install llama-cpp-python

from unsloth import FastLanguageModel
import torch

# 1. LOAD LẠI MODEL TỪ ADAPTER
# Lần này tải lại từ thư mục vừa lưu
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gpt20b_lora_adapter", 
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

# 2. EXPORT GGUF
# Quá trình này tốn rất nhiều RAM. Nếu nó báo lỗi "Killed", bạn phải dùng cách dự phòng bên dưới.
try:
    model.save_pretrained_gguf(
        "model_gguf", 
        tokenizer, 
        quantization_method = "q4_k_m"
    )
    print("Export GGUF THÀNH CÔNG!")
except Exception as e:
    print(f"Export thất bại do thiếu RAM: {e}")
    print("Xem hướng dẫn 'Dự phòng' dưới đây.")

🔴 **DỰ PHÒNG CHO CÁCH 1:**
Nếu Colab báo lỗi "Killed" lúc Export (do hết RAM 12GB):

Vào mục File bên trái Colab.
Chuột phải vào thư mục gpt20b_lora_adapter -> Download.
Về máy tính nhà (nơi có RAM 16GB+), tải Llama.cpp bản Windows, dùng lệnh merge-lora để gộp file adapter này với model gốc openai/gpt-oss-20b để tạo ra GGUF. Đây là cách duy nhất chắc chắn 100% thành công cho model 20B nếu Colab yếu.

## **CÁCH 2: Chuyển sang Model nhỏ hơn (Khuyên dùng - Ổn định, Tốc độ cao)**

Với cấu hình T4 và 12GB RAM, model ~8B là điểm sáng nhất. Nó đủ thông minh cho các tác vụ kiểm toán chuyên ngành, chạy nhanh và chắc chắn export thành công.

Tôi chọn model: **unsloth/Llama-3.1-8B-bnb-4bit.**

Kích thước GGUF: ~5.5GB (Dễ tải, dễ chạy).
Hiệu năng: Rất thông minh, hiện đại.
Code đầy đủ cho Cách 2:

In [ ]:
%%capture
# 1. CÀI ĐẶT
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes
!pip install llama-cpp-python

import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# 2. LOAD MODEL 8B
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# --- SỬA LỖI TOKENIZER TEMPLATE ---
# Gán template mặc định cho Llama 3.1 nếu bị thiếu
if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% set loop_messages = messages %}{% for message in loop_messages %}{% if message['role'] == 'user' %}{{ '<|start_header_id|>user<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}{% elif message['role'] == 'assistant' %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"

# 3. CẤU HÌNH LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
)

# 4. LOAD DỮ LIỆU (CẢ 2 FILES 2024 VÀ 2025)
try:
    # THAY ĐỔI Ở ĐÂY: Truyền danh sách tên file
    dataset = load_dataset(
        "json", 
        data_files=["audit-card2024.jsonl", "audit-card2025.jsonl"], 
        split="train"
    )
    
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"Đã tải dữ liệu thành công! Tổng số mẫu: {len(dataset)}")

except Exception as e:
    print(f"Lỗi khi tải file: {e}")
    print("Đang sử dụng dữ liệu giả để test...")
    
    # Dữ liệu giả
    dummy_data = [
        {"messages": [{"role": "system", "content": "Bạn là chuyên gia thẻ ngân hàng."}, {"role": "user", "content": "Tại sao cần kiểm tra inactive accounts?"}, {"role": "assistant", "content": "Để giảm thiểu rủi ro bảo mật và gian lận."}]}
    ]
    dataset = Dataset.from_list(dummy_data)
    
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. TRAINING
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Bạn có thể tăng số bước này lên nếu dataset lớn
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()

# 6. EXPORT GGUF
print("Bắt đầu export GGUF...")
model.save_pretrained_gguf(
    "llama31_8b_card_audit",
    tokenizer, 
    quantization_method = "q4_k_m"
)
print("XONG! File nằm trong thư mục llama31_8b_card_audit")

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
# Nếu đã kết nối rồi thì nó sẽ báo "Mounted at /content/drive"
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo ở Colab
source_dir = "/content/llama31_8b_card_audit_gguf"

# Đích: Thư mục bạn muốn lưu trên Google Drive (tên tùy bạn đặt)
target_dir = "/content/drive/MyDrive/AI_Models_CardAudit_8B"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
# Dùng lệnh '!cp -r' (copy recursive) của Linux để copy cả thư mục
# Đây là cách nhanh nhất và ổn định nhất cho file lớn trên Colab
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("Quá trình này có thể mất từ 5 đến 15 phút tùy tốc độ mạng, vui lòng không tắt tab trình duyệt.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! File đã nằm trong thư mục AI_Models_CardAudit_8B trên Google Drive của bạn.")

**TỔNG KẾT LỜI KHUYÊN:**
Nếu bạn muốn *đảm bảo 100%* thành công và file chạy mượt trên máy tính cá nhân sau này: Hãy chọn CÁCH 2. Model 8B hiện tại đủ thông minh để làm hầu hết các việc xử lý văn bản, phân tích dữ liệu kiểm toán.
Nếu bạn bắt buộc phải dùng 20B: Bạn phải chấp nhận rủi ro bị ngắt RAM. Nếu Colab không export được, ***bạn phải tải thư mục gpt20b_lora_adapter về máy tính và dùng tool llama.cpp trên Windows để merge thủ công***.

# Cách 2. Colab T4 with RAG & FineTunne LiquidAI/LFM2.5-1.2B-Thinking

In [ ]:
%%capture
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
# Phiên bản unsloth mới nhất hỗ trợ tốt LFM2.5
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Cài đặt các thư viện hỗ trợ 4-bit quantization và training
!pip install --no-deps "xformers<0.0.26" triton peft accelerate bitsandbytes
# Cài đặt llama-cpp-python để hỗ trợ export GGUF
!pip install llama-cpp-python

# Tắt wandb để tránh đăng nhập không cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# 2. LOAD MODEL LFM2.5-1.2B-THINKING
# Sử dụng dtype=None để tự động chọn bf16 hoặc fp16 tương thích với T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "LiquidAI/LFM2.5-1.2B-Thinking", 
    max_seq_length = 2048,  # Có thể tăng lên 4096 nếu cần context dài, nhưng 2048 sẽ train nhanh hơn
    dtype = None, 
    load_in_4bit = True,    # Tải dưới dạng 4-bit để tiết kiệm VRAM (dù 1.2B đã rất nhẹ)
)

# 3. CẤU HÌNH LORA CHO LFM2.5
# Lưu ý: LFM2.5 có kiến trúc hơi khác Llama, đặc biệt là module 'in_proj'
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"], # Danh sách chuẩn cho LFM2.5
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Quan trọng cho T4 để tiết kiệm bộ nhớ
    random_state = 3407,
    use_rslora = True, # Ranked LoRA giúp ổn định hơn
)

# 4. LOAD DỮ LIỆU (Audit Card 2024 & 2025 & KNIME K-AI, PBI report)
print("Đang tải dữ liệu training...")
try:
    # Load cả 2 file jsonl
    dataset = load_dataset(
        "json", 
        data_files=["audit-card2024.jsonl", "audit-card2025.jsonl", "k-ai.jsonl", "pbireport.jsonl"], 
        split="train"
    )
    
    # Chuẩn bị định dạng ShareGPT cho Unsloth
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    
    # Hàm định dạng prompt sử dụng Chat Template mặc định của LFM2.5 (ChatML-like)
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        # LFM2.5 dùng ChatML, apply_chat_template sẽ tự xử lý đúng định dạng
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"✅ Đã tải dữ liệu thành công! Tổng số mẫu: {len(dataset)}")

except Exception as e:
    print(f"⚠️ Lỗi khi tải file: {e}")
    print("Đang sử dụng dữ liệu giả (dummy data) để test cấu hình...")
    
    # Dữ liệu giả định dạng giống thật để test luồng
    dummy_data = [
        {
            "messages": [
                {"role": "system", "content": "Bạn là chuyên gia tư vấn audit thẻ ngân hàng."}, 
                {"role": "user", "content": "Tại sao cần kiểm tra inactive accounts?"}, 
                {"role": "assistant", "content": "Để giảm thiểu rủi ro bảo mật, gian lận và tuân thủ quy định Ngân hàng."}
            ]
        }
    ]
    dataset = Dataset.from_list(dummy_data)
    
    from unsloth.chat_templates import standardize_sharegpt
    dataset = standardize_sharegpt(dataset)
    
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts, }
    
    dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. TRAINING (SFT)
# Với model 1.2B, bạn có thể tăng max_steps hoặc num_train_epochs tùy ý mà không lo quá tải
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2, # T4 15GB có thể nhồi lên tới 4, nhưng 2 là an toàn
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, 
        # Nếu dataset của bạn lớn (hàng ngàn mẫu), hãy đổi max_steps thành: num_train_epochs = 1 hoặc 2
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        dataset_text_field = "text", # Quan trọng: trỏ đến trường text đã format
    ),
)
print("🚀 Bắt đầu training...")
trainer.train()

# 6. LƯU MODEL (LoRA adapters)
print("💾 Đang lưu LoRA adapters...")
model.save_pretrained("lfm25_thinking_lora") 
tokenizer.save_pretrained("lfm25_thinking_lora")

# 7. EXPORT GGUF (Để chạy trên LM Studio/Ollama)
print("🔄 Bắt đầu export GGUF (đây là bước mất thời gian nhất)...")
# Unsloth sẽ tự merge LoRA vào base model rồi quantize
model.save_pretrained_gguf(
    "lfm25_thinking_gguf", 
    tokenizer, 
    quantization_method = "q4_k_m" # Cân bằng giữa tốc độ và độ chính xác
)
print("✅ XONG! File GGUF đã nằm trong thư mục: /content/lfm25_thinking_gguf")

Đoạn Code Copy sang Google Drive (Đã cập nhật tên thư mục)

Đoạn code này tương tự nhưng tôi đã cập nhật đường dẫn nguồn và đích để khớp với tên model mới:

In [ ]:
from google.colab import drive
import os

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Nguồn: Thư mục chứa file GGUF vừa tạo
source_dir = "/content/lfm25_thinking_gguf"

# Đích: Thư mục trên Google Drive (Đã đổi tên để phân biệt với model cũ)
target_dir = "/content/drive/MyDrive/AI_Models_LFM25_Thinking_Audit"

# 3. Tạo thư mục đích nếu chưa có
os.makedirs(target_dir, exist_ok=True)

# 4. Thực hiện lệnh Copy
print(f"Đang bắt đầu copy từ {source_dir} sang Google Drive...")
print("⏳ Quá trình này có thể mất từ 2 đến 5 phút vì model 1.2B khá nhẹ.")

# Thực hiện lệnh copy
!cp -r {source_dir} {target_dir}

print("\n>>> HOÀN TẤT! Model LFM2.5-Thinking đã sẵn sàng trong thư mục AI_Models_LFM25_Thinking_Audit trên Google Drive.")